In [15]:
import requests
import pandas as pd


In [3]:
url = "https://raw.githubusercontent.com/huggingface/transformers/refs/heads/main/MIGRATION_GUIDE_V5.md"

In [4]:
# Downloading
response = requests.get(url)
response.raise_for_status()

whole_text = response.text

print("Characters:", len(whole_text))
print(whole_text[:1000])

Characters: 44657
<!---
Copyright 2025 The HuggingFace Team. All rights reserved.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# Version 5 Migration guide

## Library-wide changes with widespread impact

### Removal of TensorFlow and Jax

We're removing the TensorFlow and Jax parts of the library. This will help us focus fully on `torch`
going forward and will greatly reduce the maintenance cost of models. We are working with tools from
the Jax ecosystem still (such as MaxText) in order to see how we can remai

In [5]:
headings = []
code = False                      # initially looking for just the # character was giving code comments too

for line in whole_text.splitlines():
  if line.strip().startswith("```"):
    if code == True:
      code = False
    else:
      code = True
    continue
  if not code and line.startswith("#"):
    headings.append(line)

print("Heading numbers:", len(headings))

for heading in headings:
  print(heading)

Heading numbers: 58
# Version 5 Migration guide
## Library-wide changes with widespread impact
### Removal of TensorFlow and Jax
### Dynamic weight loading
## Tokenization
### Backend Architecture Changes: moving away from the slow/fast tokenizer separation
### Defining a tokenizers outside of the existing backends
### API Changes
#### 1. Direct tokenizer initialization with vocab and merges
#### 2. Simplified decoding API
#### 3. Unified encoding API
#### 4. `apply_chat_template` returns `BatchEncoding`
#### 5. Removed legacy configuration file saving:
#### 6. Model-Specific Changes
### Testing Changes
#### 7. Deprecated / Modified Features
## Disclaimers for the RC0
### PEFT + MoE:
### Tensor parallel and Expert parallel + MoE
### Remote code incompatibility
### Custom pretrained models:
## Library-wide changes with lesser impact
### Drop support for `safe_serialization=False`
### 50GB default shard size
### `use_auth_token`
### Attention-related features
### Updates to supported tor

In [11]:
sections = []
main_section = None
subsection = None
subsubsection = None
current = []
code = False
title = None

for line in whole_text.splitlines():
  stripped = line.strip()
  if stripped.startswith("```"):
    if not code:
      code = True
    else:
      code = False
    current.append(line)
    continue
  if code:
    current.append(line)
    continue
  if line.startswith("#### "):
    text = "\n".join(current).strip()
    if text:
      sections.append({
          "main_section": main_section,
          "subsection": subsection,
          "subsubsection": subsubsection,
          "text": text
      })
    subsubsection = line[5:].strip()
    current = []
  elif line.startswith("### "):
    text = "\n".join(current).strip()
    if text:
      sections.append({
          "main_section": main_section,
          "subsection": subsection,
          "subsubsection": subsubsection,
          "text": text
      })
    subsection = line[4:].strip()
    subsubsection = None
    current = []
  elif line.startswith("## "):
    text = "\n".join(current).strip()
    if text:
      sections.append({
          "main_section": main_section,
          "subsection": subsection,
          "subsubsection": subsubsection,
          "text": text
      })
    main_section = line[3:].strip()
    subsection = None
    subsubsection = None
    current = []
  elif line.startswith("# "):
    title = line[2:].strip()
  else:
    current.append(line)

text = "\n".join(current).strip()

if text:
  sections.append({
      "main_section": main_section,
      "subsection": subsection,
      "subsubsection": subsubsection,
      "text": text
  })



for i, section in enumerate(sections):
    print(f"SECTION {i + 1}")

    print("Main:", section["main_section"])
    print("Sub:", section["subsection"])
    print("Sub-sub:", section["subsubsection"])

    print("Text:")
    print(section["text"])

    print("-" * 80)

SECTION 1
Main: None
Sub: None
Sub-sub: None
Text:
<!---
Copyright 2025 The HuggingFace Team. All rights reserved.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->
--------------------------------------------------------------------------------
SECTION 2
Main: Library-wide changes with widespread impact
Sub: Removal of TensorFlow and Jax
Sub-sub: None
Text:
We're removing the TensorFlow and Jax parts of the library. This will help us focus fully on `torch`
going forward and will greatly reduce the maintenance cost 

In [13]:
print("Title:", title)
print("Number of records parsed:", len(sections))
print("First section:", sections[1]["main_section"])
print("Last section:", sections[-1]["main_section"])

Title: Version 5 Migration guide
Number of records parsed: 51
First section: Library-wide changes with widespread impact
Last section: Requirements update


In [19]:
df = pd.DataFrame(sections)

df.head()
print(df.shape)
print(df.isna().sum())
df = df[df["main_section"].notna()].copy()
print(df.shape)
df.head()

(51, 4)
main_section      1
subsection        9
subsubsection    39
text              0
dtype: int64
(50, 4)


,main_section,subsection,subsubsection,text
1,Library-wide changes with widespread impact,Removal of TensorFlow and Jax,None,We're removing the TensorFlow and Jax parts of...
2,Library-wide changes with widespread impact,Dynamic weight loading,None,We introduce a new weight loading API in `tran...
3,Tokenization,None,None,Just as we moved towards a single backend libr...
4,Tokenization,Backend Architecture Changes: moving away from...,None,"Up to now, transformers maintained two paralle..."
5,Tokenization,Defining a tokenizers outside of the existing ...,None,We enable users and tokenizer builders to defi...


In [20]:
df["version"] = "v5"
df["document_type"] = "migration_guide"
df["source_url"] = url
print(df.shape)
df.head()

(50, 7)


,main_section,subsection,subsubsection,text,version,document_type,source_url
1,Library-wide changes with widespread impact,Removal of TensorFlow and Jax,None,We're removing the TensorFlow and Jax parts of...,v5,migration_guide,https://raw.githubusercontent.com/huggingface/...
2,Library-wide changes with widespread impact,Dynamic weight loading,None,We introduce a new weight loading API in `tran...,v5,migration_guide,https://raw.githubusercontent.com/huggingface/...
3,Tokenization,None,None,Just as we moved towards a single backend libr...,v5,migration_guide,https://raw.githubusercontent.com/huggingface/...
4,Tokenization,Backend Architecture Changes: moving away from...,None,"Up to now, transformers maintained two paralle...",v5,migration_guide,https://raw.githubusercontent.com/huggingface/...
5,Tokenization,Defining a tokenizers outside of the existing ...,None,We enable users and tokenizer builders to defi...,v5,migration_guide,https://raw.githubusercontent.com/huggingface/...


In [26]:
def make_sections(row):
  parts = [row["main_section"], row["subsection"], row["subsubsection"]]
  cleaned = []

  for part in parts:
    if pd.notna(part):
      cleaned.append(part)

  return " > ".join(cleaned)

In [27]:
df["section_path"] = df.apply(make_sections, axis=1)
df[["section_path", "text"]].head(10)

,section_path,text
1,Library-wide changes with widespread impact > ...,We're removing the TensorFlow and Jax parts of...
2,Library-wide changes with widespread impact > ...,We introduce a new weight loading API in `tran...
3,Tokenization,Just as we moved towards a single backend libr...
4,Tokenization > Backend Architecture Changes: m...,"Up to now, transformers maintained two paralle..."
5,Tokenization > Defining a tokenizers outside o...,We enable users and tokenizer builders to defi...
6,Tokenization > API Changes > 1. Direct tokeniz...,"Starting with v5, we now enable initializing b..."
7,Tokenization > API Changes > 2. Simplified dec...,The `batch_decode` and `decode` methods have b...
8,Tokenization > API Changes > 3. Unified encodi...,The `encode_plus` method is deprecated in favo...
9,Tokenization > API Changes > 4. `apply_chat_te...,"Previously, `apply_chat_template` returned `in..."
10,Tokenization > API Changes > 5. Removed legacy...,We simplify the serialization of tokenization ...


In [28]:
df["char_count"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()

df[["section_path", "char_count", "word_count"]].sort_values("word_count", ascending=False).head(10)

,section_path,char_count,word_count
17,Disclaimers for the RC0 > Custom pretrained mo...,3226,314
2,Library-wide changes with widespread impact > ...,2258,313
13,Tokenization > Testing Changes > 7. Deprecated...,2562,248
6,Tokenization > API Changes > 1. Direct tokeniz...,1618,219
47,CLI,1459,201
3,Tokenization,1822,195
23,Library-wide changes with lesser impact > Feat...,1665,194
4,Tokenization > Backend Architecture Changes: m...,1640,186
5,Tokenization > Defining a tokenizers outside o...,1318,186
26,Configuration,1420,179


In [29]:
df.to_csv("migration_guide_sections.csv", index=False)